In [0]:
import pandas as pd

In [0]:
# Read from Unity Catalog table and convert to pandas
patient_data_df = spark.table('workspace.default.diabetic_data').toPandas()

In [0]:
patient_data_df.head()

,admission_type_id,description,encounter_id,patient_nbr,race,gender,age,weight,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,6,None,2278392.0,8222157.0,Caucasian,Female,[0-10),?,25.0,1.0,1.0,?,Pediatrics-Endocrinology,41.0,0.0,1.0,0.0,0.0,0.0,250.83,?,?,1.0,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO
1,1,None,149190.0,55629189.0,Caucasian,Female,[10-20),?,1.0,7.0,3.0,?,?,59.0,0.0,18.0,0.0,0.0,0.0,276,250.01,255,9.0,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,>30
2,1,None,64410.0,86047875.0,AfricanAmerican,Female,[20-30),?,1.0,7.0,2.0,?,?,11.0,5.0,13.0,2.0,0.0,1.0,648,250,V27,6.0,None,None,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO
3,1,None,500364.0,82442376.0,Caucasian,Male,[30-40),?,1.0,7.0,2.0,?,?,44.0,1.0,16.0,0.0,0.0,0.0,8,250.43,403,7.0,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO
4,1,None,16680.0,42519267.0,Caucasian,Male,[40-50),?,1.0,7.0,1.0,?,?,51.0,0.0,8.0,0.0,0.0,0.0,197,157,250,5.0,None,None,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [0]:
from pyspark.sql import functions as F

# Bronze source
bronze_df = spark.table("workspace.default.diabetic_data")

silver_df = (
    bronze_df
    .select(
        F.col("patient_nbr").cast("string").alias("patient_id"),
        F.col("encounter_id").cast("string").alias("encounter_id"),
        F.col("age").cast("string").alias("age"),
        F.col("time_in_hospital").alias("time_in_hospital"),
        F.col("readmitted").cast("string").alias("readmitted"),
        F.col("diag_1").cast("string").alias("diag_1"),
        F.col("admission_type_id").alias("admission_type_id"),
        F.col("discharge_disposition_id").alias("discharge_disposition_id"),
    )
    .withColumn("patient_id", F.trim("patient_id"))
    .withColumn("encounter_id", F.trim("encounter_id"))
    .withColumn("age", F.trim("age"))
    .withColumn("readmitted", F.upper(F.trim("readmitted")))
    .withColumn("diag_1", F.trim("diag_1"))
    .withColumn("patient_id", F.when((F.col("patient_id") == "") | (F.col("patient_id") == "?"), None).otherwise(F.col("patient_id")))
    .withColumn("encounter_id", F.when((F.col("encounter_id") == "") | (F.col("encounter_id") == "?"), None).otherwise(F.col("encounter_id")))
    .withColumn("age", F.when((F.col("age") == "") | (F.col("age") == "?"), None).otherwise(F.col("age")))
    .withColumn("diag_1", F.when((F.col("diag_1") == "") | (F.col("diag_1") == "?"), None).otherwise(F.col("diag_1")))
    .withColumn(
        "age",
        F.when(F.col("age").isNull(), None)
         .otherwise(F.regexp_replace(F.regexp_replace(F.col("age"), r"^\[", ""), r"\)$", ""))
    )
    .withColumn(
        "readmitted",
        F.when(F.col("readmitted").isin("NO", "<30", ">30"), F.col("readmitted")).otherwise(None)
    )
    .withColumn("time_in_hospital", F.col("time_in_hospital").cast("int"))
    .withColumn("admission_type_id", F.col("admission_type_id").cast("int"))
    .withColumn("discharge_disposition_id", F.col("discharge_disposition_id").cast("int"))
    .filter(F.col("patient_id").isNotNull())
    .filter(F.col("encounter_id").isNotNull())
    .filter(F.col("time_in_hospital").isNotNull() & (F.col("time_in_hospital") >= 0))
    .filter(F.col("admission_type_id").isNotNull() & (F.col("admission_type_id") > 0))
    .filter(F.col("discharge_disposition_id").isNotNull() & (F.col("discharge_disposition_id") > 0))
    .dropDuplicates(["encounter_id"])
)

In [0]:
silver_row_count = silver_df.count()
print(f"Silver dataset row count: {silver_row_count}")
print(f"Silver dataset column count: {len(silver_df.columns)}")

Silver dataset row count: 101766
Silver dataset column count: 8


In [0]:
silver_df.head(10)

[Row(patient_id='8222157', encounter_id='2278392', age='0-10', time_in_hospital=1, readmitted='NO', diag_1='250.83', admission_type_id=6, discharge_disposition_id=25),
 Row(patient_id='55629189', encounter_id='149190', age='10-20', time_in_hospital=3, readmitted='>30', diag_1='276', admission_type_id=1, discharge_disposition_id=1),
 Row(patient_id='86047875', encounter_id='64410', age='20-30', time_in_hospital=2, readmitted='NO', diag_1='648', admission_type_id=1, discharge_disposition_id=1),
 Row(patient_id='82442376', encounter_id='500364', age='30-40', time_in_hospital=2, readmitted='NO', diag_1='8', admission_type_id=1, discharge_disposition_id=1),
 Row(patient_id='42519267', encounter_id='16680', age='40-50', time_in_hospital=1, readmitted='NO', diag_1='197', admission_type_id=1, discharge_disposition_id=1),
 Row(patient_id='82637451', encounter_id='35754', age='50-60', time_in_hospital=3, readmitted='>30', diag_1='414', admission_type_id=2, discharge_disposition_id=1),
 Row(patie

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")

(
    silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.silver.patient_encounters")
)

spark.sql("SELECT COUNT(*) AS row_count FROM workspace.silver.patient_encounters").show()
spark.sql("SELECT DISTINCT readmitted FROM workspace.silver.patient_encounters ORDER BY readmitted").show()
spark.sql("SELECT DISTINCT age FROM workspace.silver.patient_encounters ORDER BY age LIMIT 20").show(truncate=False)
spark.sql("DESCRIBE TABLE workspace.silver.patient_encounters").show(truncate=False)

+---------+
|row_count|
+---------+
|   101766|
+---------+

+----------+
|readmitted|
+----------+
|       <30|
|       >30|
|        NO|
+----------+

+------+
|age   |
+------+
|0-10  |
|10-20 |
|20-30 |
|30-40 |
|40-50 |
|50-60 |
|60-70 |
|70-80 |
|80-90 |
|90-100|
+------+

+------------------------+---------+-------+
|col_name                |data_type|comment|
+------------------------+---------+-------+
|patient_id              |string   |NULL   |
|encounter_id            |string   |NULL   |
|age                     |string   |NULL   |
|time_in_hospital        |int      |NULL   |
|readmitted              |string   |NULL   |
|diag_1                  |string   |NULL   |
|admission_type_id       |int      |NULL   |
|discharge_disposition_id|int      |NULL   |
+------------------------+---------+-------+



In [0]:
ids_mapping_df = spark.table('workspace.default.ids_mapping').toPandas()

In [0]:
ids_mapping_df

,admission_type_id,description
0,1,Emergency
1,2,Urgent
2,3,Elective
3,4,Newborn
4,5,Not Available
...,...,...
62,22,Transfer from hospital inpt/same fac reslt in...
63,23,Born inside this hospital
64,24,Born outside this hospital
65,25,Transfer from Ambulatory Surgery Center


In [0]:
for row in ids_mapping_df.iterrows():
    print(row)

(0, admission_type_id            1
description          Emergency
Name: 0, dtype: object)
(1, admission_type_id         2
description          Urgent
Name: 1, dtype: object)
(2, admission_type_id           3
description          Elective
Name: 2, dtype: object)
(3, admission_type_id          4
description          Newborn
Name: 3, dtype: object)
(4, admission_type_id                5
description          Not Available
Name: 4, dtype: object)
(5, admission_type_id       6
description          NULL
Name: 5, dtype: object)
(6, admission_type_id                7
description          Trauma Center
Name: 6, dtype: object)
(7, admission_type_id             8
description          Not Mapped
Name: 7, dtype: object)
(8, admission_type_id    None
description          None
Name: 8, dtype: object)
(9, admission_type_id    discharge_disposition_id
description                       description
Name: 9, dtype: object)
(10, admission_type_id                     1
description          Discharged to home

In [ ]:
import pandas as pd
from pyspark.sql import functions as F

# Cleanup raw ids_mapping into 3 clean mapping datasets
ids_mapping_pdf = spark.table("workspace.default.ids_mapping").toPandas()
ids_mapping_pdf = ids_mapping_pdf.rename(columns={"admission_type_id": "raw_id", "description": "description"})
ids_mapping_pdf["raw_id"] = ids_mapping_pdf["raw_id"].astype("string").str.strip()
ids_mapping_pdf["description"] = ids_mapping_pdf["description"].astype("string").str.strip()

section_switch = {
    "discharge_disposition_id": "discharge_disposition",
    "admission_source_id": "admission_source",
}

current_section = "admission_type"
admission_type_rows = []
discharge_disposition_rows = []
admission_source_rows = []

for _, row in ids_mapping_pdf.iterrows():
    raw_id = None if pd.isna(row["raw_id"]) else str(row["raw_id"]).strip()
    desc = None if pd.isna(row["description"]) else str(row["description"]).strip()

    # Skip empty separators and stringified null rows
    if raw_id in (None, "", "None", "<NA>") and desc in (None, "", "None", "<NA>"):
        continue

    # Section header rows embedded in data
    if raw_id in section_switch and desc == "description":
        current_section = section_switch[raw_id]
        continue

    # Skip malformed/header-like rows
    if raw_id in ("admission_type_id", "description", "discharge_disposition_id", "admission_source_id"):
        continue
    if not str(raw_id).isdigit():
        continue

    rec = {"id": int(raw_id), "description": desc if desc not in ("<NA>", "None") else None}

    if current_section == "admission_type":
        admission_type_rows.append(rec)
    elif current_section == "discharge_disposition":
        discharge_disposition_rows.append(rec)
    elif current_section == "admission_source":
        admission_source_rows.append(rec)

admission_type_pdf = pd.DataFrame(admission_type_rows).drop_duplicates(subset=["id"]).sort_values("id")
discharge_disposition_pdf = pd.DataFrame(discharge_disposition_rows).drop_duplicates(subset=["id"]).sort_values("id")
admission_source_pdf = pd.DataFrame(admission_source_rows).drop_duplicates(subset=["id"]).sort_values("id")

admission_type_df = (
    spark.createDataFrame(admission_type_pdf)
    .withColumnRenamed("id", "admission_type_id")
    .withColumn("admission_type_id", F.col("admission_type_id").cast("int"))
    .withColumn("description", F.col("description").cast("string"))
)

discharge_disposition_df = (
    spark.createDataFrame(discharge_disposition_pdf)
    .withColumnRenamed("id", "discharge_disposition_id")
    .withColumn("discharge_disposition_id", F.col("discharge_disposition_id").cast("int"))
    .withColumn("description", F.col("description").cast("string"))
)

admission_source_df = (
    spark.createDataFrame(admission_source_pdf)
    .withColumnRenamed("id", "admission_source_id")
    .withColumn("admission_source_id", F.col("admission_source_id").cast("int"))
    .withColumn("description", F.col("description").cast("string"))
)

In [ ]:
# Verify cleaned mapping outputs before writing
print("admission_type rows:", admission_type_df.count())
print("discharge_disposition rows:", discharge_disposition_df.count())
print("admission_source rows:", admission_source_df.count())

print("\nDistinct ID counts (should match row counts):")
print("admission_type distinct IDs:", admission_type_df.select("admission_type_id").distinct().count())
print("discharge_disposition distinct IDs:", discharge_disposition_df.select("discharge_disposition_id").distinct().count())
print("admission_source distinct IDs:", admission_source_df.select("admission_source_id").distinct().count())

print("\nSample rows:")
admission_type_df.orderBy("admission_type_id").show(10, truncate=False)
discharge_disposition_df.orderBy("discharge_disposition_id").show(10, truncate=False)
admission_source_df.orderBy("admission_source_id").show(10, truncate=False)

In [ ]:
# Write cleaned mapping tables to silver and read back to verify
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")

admission_type_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.silver.admission_type_mapping")
discharge_disposition_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.silver.discharge_disposition_mapping")
admission_source_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.silver.admission_source_mapping")

print("Written tables:")
print("- workspace.silver.admission_type_mapping")
print("- workspace.silver.discharge_disposition_mapping")
print("- workspace.silver.admission_source_mapping")

print("\nRead-back verification:")
spark.sql("SELECT COUNT(*) AS row_count FROM workspace.silver.admission_type_mapping").show()
spark.sql("SELECT COUNT(*) AS row_count FROM workspace.silver.discharge_disposition_mapping").show()
spark.sql("SELECT COUNT(*) AS row_count FROM workspace.silver.admission_source_mapping").show()

spark.sql("SELECT * FROM workspace.silver.admission_type_mapping ORDER BY admission_type_id LIMIT 10").show(truncate=False)
spark.sql("SELECT * FROM workspace.silver.discharge_disposition_mapping ORDER BY discharge_disposition_id LIMIT 10").show(truncate=False)
spark.sql("SELECT * FROM workspace.silver.admission_source_mapping ORDER BY admission_source_id LIMIT 10").show(truncate=False)